**Imports and Utility**

In [10]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,TensorDataset
from torchvision import datasets,transforms
from sklearn.metrics import roc_auc_score,average_precision_score,precision_recall_curve

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(0)

In [11]:
def load_mnist():
    trfm = transforms.Compose([
        (transforms.ToTensor())
    ])
    train = datasets.MNIST(root='./data',train=True,download=True,transform=trfm)
    test = datasets.MNIST(root='./data',train=False,download=True,transform=trfm)
    return train,test

def flatten_dataset(ds):
    loader = DataLoader(ds,batch_size=len(ds))
    for X,y in loader:
        return X.flatten(1),y

**Model Architectures**

In [12]:
class ClassifierWithDecoder(nn.Module):
    def __init__(self,input_dim=784,hidden=256,n_classes=10):
        super().__init__()
        
        self.common = nn.Sequential(
            nn.Linear(input_dim,hidden),
            nn.GELU(),
            nn.Linear(hidden,hidden),
            nn.GELU()
        )
        
        self.classifier = nn.Linear(hidden,n_classes)
        
        self.decoder = nn.Sequential(
            nn.Linear(hidden,hidden),
            nn.GELU(),
            nn.Linear(hidden,input_dim),
            nn.GELU(),
            nn.Sigmoid() # Squish to [0,1] to compare with processed version of input
        )
    
    def forward(self,x):
        feats = self.common(x)
        
        logits = self.classifier(feats)
        probs = F.softmax(logits,dim=1)
        
        reconstr = self.decoder(feats)
        return logits,probs,reconstr,feats

In [13]:
class AbnormalityModule(nn.Module):
    def __init__(self,feat_dim=256,input_dim=784,n_classes=10,hidden=256):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(feat_dim+input_dim+n_classes,hidden), # Combined all 3 because the paper's figure showed so
            nn.GELU(),
            nn.Linear(hidden,hidden),
            nn.GELU(),
            nn.Linear(hidden,1)
        )
        
    def forward(self,feats,probs,sq_error):
        z = torch.cat([feats,probs,sq_error],dim=1)
        return torch.sigmoid(self.net(z)).squeeze(1)

**Noise Addition Functions**

In [14]:
# Functions for Batch of data -> Batch of noisy data 

def add_gaussian_noise(x,std=0.5):
    return torch.clamp(x+std*torch.randn_like(x),0,1)

def add_uniform_noise(x,noise_level=0.1):
    noise = (torch.rand_like(x)*2-1) * noise_level
    return torch.clamp(x+noise,0,1)

def blur_batch(x_flat,kernel_size=3):
    x = x_flat.view(-1,1,28,28)
    pad = kernel_size//2
    x = F.avg_pool2d(F.pad(x,[pad]*4,mode="replicate"),kernel_size=kernel_size,stride=1)
    return x.view(x.shape[0],-1)

def rotate_batch(x_flat,degrees=45):
    import torchvision.transforms.functional as tf
    x = x_flat.view(-1,1,28,28)
    out = torch.empty_like(x)
    for i in range(x.shape[0]):
        angle = float(np.random.uniform(-degrees,degrees))
        out[i]=tf.rotate(x[i],angle)
    return out.view(out.shape[0],-1)

def make_abnormal_batch(x_flat):
    choice = np.random.randint(4)
    if choice == 0:
        return add_gaussian_noise(x_flat)
    if choice == 1:
        return add_uniform_noise(x_flat)
    if choice == 2:
        return blur_batch(x_flat)
    else:
        return rotate_batch(x_flat)

In [15]:
def train_classifier_decoder(model,X_train,y_train,epochs=15,batch_size=128,lr=1e-3):
    model.to(device)
    model.train()
    
    optimizer = torch.optim.Adam(model.parameters(),lr=lr)
    ds = TensorDataset(X_train,y_train)
    loader = DataLoader(ds,batch_size=batch_size,shuffle=True)
    
    for epoch in range(epochs):
        total_ce,total_recon,n = 0,0,0
        
        for X,y in loader:
            X = X.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            logits,probs,reconstr,feats = model(X)
            ce_loss = F.cross_entropy(logits,y)
            recon_loss = F.mse_loss(reconstr,X)
            loss = ce_loss+recon_loss
            loss.backward()
            optimizer.step()
            
            total_ce+=ce_loss.item()*X.shape[0]
            total_recon+=recon_loss.item()*X.shape[0]
            n+=X.shape[0]
            
        print(f"[classifier+decoder] epoch {epoch + 1}/{epochs} ce={total_ce / n:.4f} recon={total_recon / n:.4f}")
    return model

In [16]:
def train_abnormality_module(clf_decoder,ab_module,X_train,y_train,epochs=2,batch_size=128,lr=1e-3):
    clf_decoder.eval()
    for param in clf_decoder.parameters():
        param.requires_grad = False
    
    ab_module.to(device)
    ab_module.train()
    
    optimizer = torch.optim.Adam(ab_module.parameters(),lr=lr)
    ds = TensorDataset(X_train)
    loader = DataLoader(ds,batch_size=batch_size,shuffle=True)
    
    for epoch in range(epochs):
        total_loss,n = 0,0
        
        for (X,) in loader:
            X = X.to(device)
            X_abnormal = make_abnormal_batch(X.cpu().to(device))
            
            X_modified = torch.cat([X,X_abnormal],dim=0)
            labels = torch.cat([torch.zeros(X.shape[0]),torch.ones(X_abnormal.shape[0])]).to(device) # Clean:0 Noise:1
            
            with torch.no_grad():
                logits,probs,reconstr,feats = clf_decoder(X_modified)
                sq_error = (reconstr - X_modified)**2
                
            optimizer.zero_grad()
            scores = ab_module(feats,probs,sq_error)
            loss = F.binary_cross_entropy(scores,labels)
            loss.backward()
            optimizer.step()
            
            total_loss+=loss.item()*X_modified.shape[0]
            n+=X_modified.shape[0]
        print(f"[abnormality module] epoch {epoch + 1}/{epochs} bce={total_loss / n:.4f}")
    return ab_module

In [17]:
@torch.no_grad()
def msp_scores(model,X):
    # Maximum Softmax Probability
    model.eval()
    logits,probs,reconstr,feats = model(X.to(device))
    return probs.max(dim=1).values.cpu().numpy()

@torch.no_grad()
def abnormality_scores(clf_decoder,ab_module,X):
    # Abnormality Module Score
    clf_decoder.eval()
    ab_module.eval()
    X = X.to(device)
    logits,probs,reconstr,feats = clf_decoder(X)
    sq_error = (reconstr-X)**2
    return ab_module(feats,probs,sq_error).cpu().numpy()

def evaluate_ood(in_scores,out_scores,higher_is_in_dist=True):
    if not higher_is_in_dist:
        in_scores,out_scores = -in_scores,-out_scores
 
    labels = np.concatenate([np.ones_like(in_scores),np.zeros_like(out_scores)])
    scores = np.concatenate([in_scores,out_scores])
 
    auroc = roc_auc_score(labels,scores)
    aupr_in = average_precision_score(labels,scores)
    aupr_out = average_precision_score(1-labels,-scores)
    return {"AUROC": auroc, "AUPR_In": aupr_in, "AUPR_Out": aupr_out}

In [18]:
train_ds,test_ds = load_mnist()
train_x,train_y = flatten_dataset(train_ds)
test_x,test_y = flatten_dataset(test_ds)

model = ClassifierWithDecoder()
model = train_classifier_decoder(model,train_x,train_y,epochs=15)

test_acc = (model(test_x.to(device))[0].argmax(1).cpu()==test_y).float().mean().item()
print(f"\nTest accuracy: {test_acc * 100:.2f}%\n")

ood_gaussian = add_gaussian_noise(test_x)
ood_uniform = add_uniform_noise(test_x)
ood_blur = blur_batch(test_x)
ood_rotate = rotate_batch(test_x[:2000])

print("--- MSP baseline ---")
in_scores = msp_scores(model,test_x)
for name,ood_x in [("Gaussian",ood_gaussian),("Uniform",ood_uniform),("Blur",ood_blur),("Rotate",ood_rotate)]:
    out_scores = msp_scores(model,ood_x)
    metrics = evaluate_ood(in_scores,out_scores,higher_is_in_dist=True)
    print(f"MNIST/{name}: {metrics}")

ab_module = AbnormalityModule()
clf_decoder = copy.deepcopy(model)
ab_module = train_abnormality_module(clf_decoder,ab_module,train_x,train_y,epochs=5)

print("\n--- Abnormality module ---")
in_scores_ab = abnormality_scores(clf_decoder,ab_module,test_x)
for name,ood_x in [("Gaussian",ood_gaussian),("Uniform",ood_uniform),("Blur",ood_blur),("Rotate",ood_rotate)]:
    out_scores_ab = abnormality_scores(clf_decoder,ab_module,ood_x)
    metrics = evaluate_ood(in_scores_ab,out_scores_ab,higher_is_in_dist=False)
    print(f"MNIST/{name}: {metrics}")

[classifier+decoder] epoch 1/15 ce=0.3317 recon=0.1963
[classifier+decoder] epoch 2/15 ce=0.1248 recon=0.1921
[classifier+decoder] epoch 3/15 ce=0.0830 recon=0.1908
[classifier+decoder] epoch 4/15 ce=0.0591 recon=0.1898
[classifier+decoder] epoch 5/15 ce=0.0446 recon=0.1893
[classifier+decoder] epoch 6/15 ce=0.0341 recon=0.1889
[classifier+decoder] epoch 7/15 ce=0.0251 recon=0.1885
[classifier+decoder] epoch 8/15 ce=0.0197 recon=0.1883
[classifier+decoder] epoch 9/15 ce=0.0150 recon=0.1881
[classifier+decoder] epoch 10/15 ce=0.0180 recon=0.1881
[classifier+decoder] epoch 11/15 ce=0.0124 recon=0.1879
[classifier+decoder] epoch 12/15 ce=0.0126 recon=0.1878
[classifier+decoder] epoch 13/15 ce=0.0069 recon=0.1875
[classifier+decoder] epoch 14/15 ce=0.0110 recon=0.1876
[classifier+decoder] epoch 15/15 ce=0.0102 recon=0.1875

Test accuracy: 97.80%

--- MSP baseline ---
MNIST/Gaussian: {'AUROC': 0.9415392, 'AUPR_In': 0.9453841497206908, 'AUPR_Out': 0.9292491359648948}
MNIST/Uniform: {'AUROC':

In [19]:
def ODIN_scores(model,X,temperature=1000,epsilon=0.0015,batch_size=256):
    model.eval()
    all_scores = []
    
    for start in range(0,len(X),batch_size):
        x = X[start:start+batch_size].to(device)
        x.requires_grad = True
        
        logits = model(x)[0]
        pred_class = (logits/temperature).argmax(dim=1)
        
        loss = F.cross_entropy(logits/temperature,pred_class)
        gradient = torch.autograd.grad(loss,x)[0]
        
        x_perturbed = (x - epsilon*gradient.sign()).clamp(0,1)
        
        with torch.no_grad():
            perturbed_logits = model(x_perturbed)[0]
            scores = F.softmax(perturbed_logits/temperature,dim=1).max(dim=1).values
        all_scores.append(scores)
    
    return torch.cat(all_scores).cpu().numpy()

def fpr_at_95tpr(id_scores,ood_scores):
    threshold = np.quantile(id_scores,0.05)
    return float(np.mean(ood_scores>=threshold)),threshold


In [20]:
def tune_odin(model,id_validation_x,ood_validation_x,temperatures=(1,10,100,1000),epsilons=(0.0,0.0005,0.001,0.0014,0.002,0.003,0.004)):
    rows = []
    for temperature in temperatures:
        for epsilon in epsilons:
            id_scores = ODIN_scores(model, id_validation_x, temperature, epsilon)
            ood_scores = ODIN_scores(model, ood_validation_x, temperature, epsilon)
            fpr, threshold = fpr_at_95tpr(id_scores, ood_scores)
            rows.append({
                "temperature": temperature,
                "epsilon": epsilon,
                "FPR@95TPR": fpr,
                "threshold": threshold,
            })

    results = pd.DataFrame(rows).sort_values("FPR@95TPR").reset_index(drop=True)
    return results, results.iloc[0].to_dict()

In [21]:
train_ds,test_ds = load_mnist()
train_x,train_y = flatten_dataset(train_ds)
test_x,test_y = flatten_dataset(test_ds)
model = ClassifierWithDecoder()
model = train_classifier_decoder(model,train_x,train_y,epochs=15)
 

validation_x = train_x[-5000:]
validation_ood_x = make_abnormal_batch(validation_x)

tuning_results, best_odin = tune_odin(model, validation_x, validation_ood_x)
print(tuning_results)
print("Selected ODIN parameters:", best_odin)

[classifier+decoder] epoch 1/15 ce=0.3289 recon=0.1970
[classifier+decoder] epoch 2/15 ce=0.1234 recon=0.1931
[classifier+decoder] epoch 3/15 ce=0.0796 recon=0.1918
[classifier+decoder] epoch 4/15 ce=0.0582 recon=0.1909
[classifier+decoder] epoch 5/15 ce=0.0431 recon=0.1904
[classifier+decoder] epoch 6/15 ce=0.0329 recon=0.1901
[classifier+decoder] epoch 7/15 ce=0.0249 recon=0.1898
[classifier+decoder] epoch 8/15 ce=0.0189 recon=0.1895
[classifier+decoder] epoch 9/15 ce=0.0173 recon=0.1894
[classifier+decoder] epoch 10/15 ce=0.0148 recon=0.1893
[classifier+decoder] epoch 11/15 ce=0.0137 recon=0.1892
[classifier+decoder] epoch 12/15 ce=0.0120 recon=0.1890
[classifier+decoder] epoch 13/15 ce=0.0075 recon=0.1888
[classifier+decoder] epoch 14/15 ce=0.0093 recon=0.1889
[classifier+decoder] epoch 15/15 ce=0.0071 recon=0.1887
    temperature  epsilon  FPR@95TPR  threshold
0          1000   0.0000     0.8646   0.101535
1          1000   0.0005     0.8654   0.101547
2           100   0.0000    

In [22]:
temperature = float(best_odin["temperature"])
epsilon = float(best_odin["epsilon"])

ood_sets = {
    "Gaussian": add_gaussian_noise(test_x),
    "Uniform": add_uniform_noise(test_x),
    "Blur": blur_batch(test_x),
    "Rotate": rotate_batch(test_x[:2000]),
}

rows = []
id_msp = msp_scores(model, test_x)
id_odin = ODIN_scores(model, test_x, temperature, epsilon)

for name, ood_x in ood_sets.items():
    for method, id_scores, out_scores in (
        ("MSP", id_msp, msp_scores(model, ood_x)),
        ("ODIN", id_odin, ODIN_scores(model, ood_x, temperature, epsilon)),
    ):
        metrics = evaluate_ood(id_scores, out_scores, higher_is_in_dist=True)
        fpr, threshold = fpr_at_95tpr(id_scores, out_scores)
        rows.append({
            "OOD set": name,
            "Method": method,
            **metrics,
            "FPR@95TPR": fpr,
            "threshold": threshold,
        })

comparison = pd.DataFrame(rows)
print(comparison.sort_values(["OOD set", "Method"]))


    OOD set Method     AUROC   AUPR_In  AUPR_Out  FPR@95TPR  threshold
4      Blur    MSP  0.630384  0.599060  0.588725     0.9246   0.987030
5      Blur   ODIN  0.690784  0.684784  0.671161     0.8480   0.101429
0  Gaussian    MSP  0.941256  0.939839  0.932172     0.3090   0.987030
1  Gaussian   ODIN  0.964722  0.962490  0.967167     0.1562   0.101429
6    Rotate    MSP  0.718434  0.908630  0.366155     0.7595   0.987030
7    Rotate   ODIN  0.728403  0.917086  0.402965     0.7240   0.101429
2   Uniform    MSP  0.581726  0.559506  0.551976     0.9350   0.987030
3   Uniform   ODIN  0.631475  0.613185  0.619420     0.8894   0.101429
